In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import poisson
from sklearn.metrics import mean_absolute_error

data = pd.read_csv(
    "data/processed/epl_enhanced_features.csv",
    parse_dates=["Date"]
)

# Existing recent-form features plus relative long-term team strength
feature_columns = [
    "home_form_points_5",
    "away_form_points_5",
    "home_avg_goals_for_5",
    "away_avg_goals_for_5",
    "home_avg_goals_against_5",
    "away_avg_goals_against_5",
    "elo_difference",
]

# Keep the same honest time split
train = data[data["Season"] != "2024-25"].copy()
test = data[data["Season"] == "2024-25"].copy()

X_train = sm.add_constant(train[feature_columns])
X_test = sm.add_constant(test[feature_columns], has_constant="add")

home_model = sm.GLM(
    train["FTHG"],
    X_train,
    family=sm.families.Poisson()
).fit()

away_model = sm.GLM(
    train["FTAG"],
    X_train,
    family=sm.families.Poisson()
).fit()

test["predicted_home_goals"] = home_model.predict(X_test)
test["predicted_away_goals"] = away_model.predict(X_test)

# Goal-prediction accuracy
home_mae = mean_absolute_error(test["FTHG"], test["predicted_home_goals"])
away_mae = mean_absolute_error(test["FTAG"], test["predicted_away_goals"])

def result_probabilities(home_lambda, away_lambda, max_goals=10):
    goals = np.arange(max_goals + 1)
    home_probs = poisson.pmf(goals, home_lambda)
    away_probs = poisson.pmf(goals, away_lambda)
    score_matrix = np.outer(home_probs, away_probs)

    return [
        np.tril(score_matrix, k=-1).sum(),
        np.trace(score_matrix),
        np.triu(score_matrix, k=1).sum(),
    ]

probabilities = np.array([
    result_probabilities(home_goals, away_goals)
    for home_goals, away_goals in zip(
        test["predicted_home_goals"],
        test["predicted_away_goals"]
    )
])

labels = np.array(["H", "D", "A"])
test["predicted_result"] = labels[probabilities.argmax(axis=1)]

# Most likely Poisson score is normally the whole number below expected goals
test["predicted_home_score"] = np.floor(
    test["predicted_home_goals"]
).astype(int)

test["predicted_away_score"] = np.floor(
    test["predicted_away_goals"]
).astype(int)

exact_score_accuracy = (
    (test["FTHG"] == test["predicted_home_score"]) &
    (test["FTAG"] == test["predicted_away_score"])
).mean()

result_accuracy = (test["FTR"] == test["predicted_result"]).mean()

print("POISSON + ELO RESULTS")
print(f"Home-goal MAE:        {home_mae:.3f}")
print(f"Away-goal MAE:        {away_mae:.3f}")
print(f"Exact-score accuracy: {exact_score_accuracy:.1%}")
print(f"Match-result accuracy:{result_accuracy:.1%}")

print("\nOriginal Poisson benchmark")
print("Home-goal MAE:         1.001")
print("Away-goal MAE:         0.893")
print("Exact-score accuracy:  11.2%")
print("Match-result accuracy: 48.5%")

POISSON + ELO RESULTS
Home-goal MAE:        0.970
Away-goal MAE:        0.886
Exact-score accuracy: 10.7%
Match-result accuracy:51.2%

Original Poisson benchmark
Home-goal MAE:         1.001
Away-goal MAE:         0.893
Exact-score accuracy:  11.2%
Match-result accuracy: 48.5%
